In [ ]:
from dotenv import load_dotenv
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.prompts import ChatPromptTemplate
import os

from rich import print as rprint
from sqlalchemy import true
from typing_extensions import Literal

######1、提供大模型#########
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_URL = os.getenv("ZHIPUAI_BASE_URL")

load_dotenv(override=True)
model = ChatZhipuAI(
    model="glm-4.5-Air",
    # api_key=ZHIPUAI_API_KEY,
    # base_url=ZHIPUAI_API_URL,
)


# 定义工具
def get_weather(city: str):
    """
    天气查询工具
    Args:
        city: 城市名称
    """
    return f"{city}天气晴朗"


# 模型和绑定
moder_with_tools = model.bind_tools([get_weather])
res = moder_with_tools.invoke(

    "今天北京天气如何"
)
rprint(res)

# 使用tool

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint


@tool("你好世界")
def get_weather(city: str, units: str = "celsius", include_forecast: bool =
True) -> str:
    """
        获取当日天气，可选择是否同时查询未来五日天气预报
        Args:
            city: 城市
            units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
            include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


res = convert_to_openai_tool(get_weather)
rprint(res)

# 自定义args_schema
## 方式1：使用Pydantic模型定义

In [4]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        description="你好世界",
        default="北京"
    )


# args_schema自定义
@tool(args_schema=WeatherInput)
def get_weather(city: str, units: str = "celsius", include_forecast: bool =
False) -> str:
    """
        获取当日天气，可选择是否同时查询未来五日天气预报

    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


res = convert_to_openai_tool(get_weather)
rprint(res)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '你好世界', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint
from typing_extensions import Literal
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        default="北京"
    )
    unit: Literal["C", "H"]  #温度
    include_forecast: bool = Field(default=False)  #天气预报


# args_schema自定义
@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = False) -> str:
    """
        获取当日天气，可选择是否同时查询未来五日天气预报

    """
    return f"{city}天气不错 温度是"


res = convert_to_openai_tool(get_weather)
rprint(res)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'type': 'string'},
                'unit': {'enum': ['C', 'H'], 'type': 'string'},
                'include_forecast': {'default': False, 'type': 'boolean'}
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

### JSON scheme

In [ ]:

from langsmith import unit
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint
from typing_extensions import Literal
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        default="北京"
    )
    unit: Literal["C", "H"]  #温度
    include_forecast: bool = Field(default=False)  #天气预报

weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}

# args_schema自定义
@tool(args_schema=weather_schema)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = False) -> str:
    """
        获取当日天气，可选择是否同时查询未来五日天气预报

    """
    temp = 22 if unit == "celsius" else 72
    if unit == "celsius":
        return f"{city}当前气温：{temp}摄氏度"
    else:
        return f"{city}当前气温：{temp}华氏度"

res = convert_to_openai_tool(get_weather)
rprint(res)